<p align="center">
  <img src="archs/Vectorless_RAG.png" alt="Vecorless_RAG" width="1200">
</p>

In [11]:
import os, json
from groq import Groq
from pageindex import PageIndexClient
import pageindex.utils as utils
from dotenv import load_dotenv

load_dotenv()

True

In [7]:
pi_client = PageIndexClient(api_key=os.getenv("PAGEINDEX_API_KEY"))
llm_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [39]:
def call_llm(prompt, model="openai/gpt-oss-20b", temperature=0):
    response = llm_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature
    )
    return response.choices[0].message.content.strip()

In [9]:
PDF_PATH = 'papers/Vectorless_RAG_paper.pdf'

print(f"📤 Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")

📤 Uploading: papers/Vectorless_RAG_paper.pdf
✅ Uploaded!
📋 Document ID: pi-cmrap6ixg023f01o3xrazyrxh


In [12]:
if pi_client.is_retrieval_ready(doc_id):
    tree = pi_client.get_tree(doc_id, node_summary=True)['result']
    print('Simplified Tree Structure of the Document:')
    utils.print_tree(tree)
else:
    print("Processing document, please try again later...")

Simplified Tree Structure of the Document:
[{'title': 'Rethinking Retrieval: From Traditional R...',
  'node_id': '0000',
  'prefix_summary': 'This paper presents a systematic evaluat...',
  'nodes': [{'title': '1 INTRODUCTION',
             'node_id': '0001',
             'summary': 'This text introduces a systematic evalua...'},
            {'title': '2 RELATED WORK',
             'node_id': '0002',
             'summary': 'This text reviews the evolution of RAG s...'},
            {'title': '3 METHOD',
             'node_id': '0003',
             'prefix_summary': '## 3 METHOD\n',
             'nodes': [{'title': '3.1 Dataset Construction and Document Pr...',
                        'node_id': '0004',
                        'summary': 'This text details the construction of an...'},
                       {'title': '3.2 Retrieval System Architectures',
                        'node_id': '0005',
                        'summary': 'This text compares two retrieval archite...'},
      

In [13]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(tree)

📚 Full Document Structure:

[0000] Rethinking Retrieval: From Traditional Retrieval Augmented Generation to Agentic and Non-Vector Reasoning Systems in the Financial Domain for Large Language Models  (p.1)
  └─ [0001] 1 INTRODUCTION  (p.1)
  └─ [0002] 2 RELATED WORK  (p.2)
  └─ [0003] 3 METHOD  (p.3)
    └─ [0004] 3.1 Dataset Construction and Document Processing  (p.3)
    └─ [0005] 3.2 Retrieval System Architectures  (p.4)
    └─ [0006] 3.3 Advanced RAG Enhancement Techniques  (p.4)
    └─ [0007] 3.4 Evaluation Framework  (p.4)
  └─ [0008] 4 EXPERIMENTS  (p.5)
    └─ [0009] 4.1 Experimental Settings  (p.5)
    └─ [0010] 4.2 Preprocessing and Node Generation  (p.5)
    └─ [0011] 4.3 Retrieval Architecture Comparison  (p.5)
    └─ [0012] 4.4 Cross-Encoder Reranking Evaluation  (p.6)
    └─ [0013] 4.5 Small-to-Big Retrieval Evaluation  (p.6)
  └─ [0014] 5 DISCUSSION  (p.6)
    └─ [0015] 5.1 Vector-Based and Hierarchical Reasoning Systems  (p.6)
    └─ [0016] 5.2 Cross-Encoder Reranking  

In [22]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(tree)
print(f"Total nodes in tree: {total}")
print("Each node = one retrievable section of the document")

Total nodes in tree: 20
Each node = one retrievable section of the document


In [73]:
def llm_tree_search(query: str, tree: list, model: str = "openai/gpt-oss-20b") -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    """

    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out

    compressed_tree = compress(tree)

    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    return call_llm(prompt=prompt, model=model)

In [ ]:
# ── Test with a sample query ─────────────────────────────────────────────────
query = "What are some Advanced RAG Enhancement Techniques?"

print(f"Query: {query}\n")
result = llm_tree_search(query, tree)

print("LLM Reasoning:")
print(json.loads(result).get("thinking", "N/A"))
print()
print("Selected Node IDs:", json.loads(result).get("node_list", []))

Query: What are some Advanced RAG Enhancement Techniques?

LLM Reasoning:
The query asks for advanced RAG enhancement techniques. The document contains a dedicated section titled "3.3 Advanced RAG Enhancement Techniques" with node_id "0006", which directly addresses this topic. Other sections (e.g., 4.4 and 4.5) evaluate these techniques but do not list them. Therefore, the most relevant node is 0006.

Selected Node IDs: ['0006']


In [52]:
node_map = utils.create_node_mapping(tree)
tree_search_result_json = json.loads(result)

print('Reasoning Process:')
utils.print_wrapped(tree_search_result_json['thinking'])

print('\nRetrieved Nodes:')
for node_id in tree_search_result_json["node_list"]:
    node = node_map[node_id]
    print(f"Node ID: {node['node_id']}\t Page: {node['page_index']}\t Title: {node['title']}")

Reasoning Process:
The query asks for advanced RAG enhancement techniques. The document contains a dedicated section
titled "3.3 Advanced RAG Enhancement Techniques" with node_id "0006", which directly addresses this
topic. Other sections (e.g., 4.4 and 4.5) evaluate these techniques but do not list them. Therefore,
the most relevant node is 0006.

Retrieved Nodes:
Node ID: 0006	 Page: 4	 Title: 3.3 Advanced RAG Enhancement Techniques


In [63]:
node_list = tree_search_result_json["node_list"]
relevant_content = "\n\n".join(node_map[node_id]["text"] for node_id in node_list)

print('Retrieved Context:\n')
utils.print_wrapped(relevant_content[:1000] + '...')

Retrieved Context:

### 3.3 Advanced RAG Enhancement Techniques

We evaluate two advanced RAG techniques independently: cross-encoder reranking for improving
retrieval precision, and small-to-big retrieval for enhancing context quality. Each technique is
assessed by comparing performance against baseline vector retrieval.

![img-0.jpeg](img-0.jpeg)

Figure 2: Small-to-big retrieval strategy. The system retrieves a target chunk at index idx through
vector search, then expands context by including neighboring chunks (idx - 2, idx - 1, idx + 1, idx
+ 2) before providing the combined context to the LLM for answer generation.

#### 3.3.1 Cross-Encoder Reranking

Cross-encoder reranking refines initial vector retrieval by reordering chunks based on fine-grained
relevance scoring (Nogueira and Cho, 2019; Sun et al., 2023). The system retrieves  \( k_{initial}
\)  chunks through vector search, then a cross-encoder model scores each chunk by jointly encoding
the query-chunk pair. The system sel

In [64]:
def find_nodes_by_ids(node_map: dict, target_ids: list) -> list:
    return [node_map[nid] for nid in target_ids if nid in node_map]


def generate_answer(query: str, nodes: list, model: str = "openai/gpt-oss-20b") -> str:
    if not nodes:
        return "No relevant sections found in the document."

    context_parts = [
        f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
        f"{node.get('text', 'Content not available.')}"
        for node in nodes
    ]
    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""

    response = call_llm(prompt=prompt, model=model)
    return response

In [68]:
# ── The complete Vectorless RAG function ─────────────────────────────────────
def vectorless_rag(query: str, tree: list, node_map: dict, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = json.loads(search_result).get("node_list", [])
    
    if verbose:
        print(f"\nReasoning: {json.loads(search_result).get('thinking', '')[:200]}...")
        print(f"Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(node_map=node_map, target_ids=node_ids)
    
    if verbose:
        print(f"Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    return answer

In [69]:
print(vectorless_rag(
    query="Give me a brief summary of the main points in the document",
    tree=tree,
    node_map=node_map,
    verbose=True
))

Query: Give me a brief summary of the main points in the document

Reasoning: The user wants a brief summary of the main points of the document. The most appropriate sections for this are the Introduction (which outlines the purpose and scope), the Discussion (which summarizes ...
Retrieved node IDs: ['0001', '0014', '0018']
Sections found: ['1 INTRODUCTION', '5 DISCUSSION', '6 CONCLUSION']
**Brief Summary**

- The study evaluates how well vector‑based agentic Retrieval‑Augmented Generation (RAG) systems perform on financial document question answering compared to hierarchical node‑based reasoning systems that use structured table‑of‑contents traversal (1 INTRODUCTION | Page 1).  
- The vector‑based baseline combines hybrid search with metadata filtering, corrective RAG, and token‑based chunking, while the hierarchical system relies on document structure without dense embeddings (1 INTRODUCTION | Page 1).  
- Two advanced RAG enhancements are tested independently on the vector system: 

In [70]:
print(vectorless_rag(
    query="Who is Shinjan?",
    tree=tree,
    node_map=node_map,
    verbose=True
))

Query: Who is Shinjan?

Reasoning: The query asks for information about a person named Shinjan. The provided document is a research paper on retrieval systems in the financial domain. Scanning the section titles and summaries, there is...
Retrieved node IDs: []
Sections found: []
No relevant sections found in the document.


I am picking this over Vector RAG for structured documents fricken' anyday!!!

It's damn good!!!

In [71]:
# — The killer feature: domain expertise without fine-tuning —
# Vector RAG needs embedding fine-tuning to encode domain knowledge (slow, costly).
# PageIndex just needs rules in the prompt — e.g. "EBITDA query → check MD&A section".
# These are routing rules that tell the LLM WHERE to look for specific queries, think of it as encoding a senior analyst's institutional knowledge.

PAPER_EXPERT_RULES = """
Route queries to the correct section using these rules:

S1  Abstract                        → high-level summary, headline numbers, win rates
S2  Introduction                    → motivation, research gap, why compare vector vs non-vector
S3  Related Work: RAG Foundations   → Lewis/Karpukhin, dense passage retrieval, naive RAG limits
S4  Related Work: Advanced RAG      → chunking, query rewriting, reranking, corrective RAG, hybrid search
S5  Related Work: Non-Vector        → hierarchical node systems, PageIndex, index-free RAG
S6  Related Work: Financial QA      → FinQA, TAT-QA, FinanceBench, 10-K/10-Q/8-K background
S7  Method: Dataset Construction    → 1,200 SEC filings, 150 QA pairs, multi-hop/single-hop/summary split
S8  Method: Hierarchical Node Trees → tree generation, GPT-4o/Gemini/GPT-4.1 mini comparison, Figure 1
S9  Method: Retrieval Architectures → vector-based agentic RAG vs hierarchical node-based system design
S10 Method: Reranking & Small-to-Big → cross-encoder reranking params, small-to-big expansion (Figure 2)
S11 Method: Evaluation Framework    → MRR, Recall@5, LLM-as-judge criteria, latency/cost measurement
S12 Experiments: Preprocessing Cost → Tables 1-3, token counts, GPT-4o/Gemini/GPT-4.1 mini cost comparison
S13 Experiments: Architecture Comparison → 68% win rate, 5.2s vs 5.98s latency, failure cases
S14 Experiments: Reranking Results  → Table 4, MRR@5 0.160→0.750, optimal (10,5) params
S15 Experiments: Small-to-Big Results → 65% win rate, async vs sync latency, per-query cost
S16 Discussion                      → why vector beat hierarchical, TOC bottleneck theory, future work
S17 Conclusion                      → final numbers, summary of all three findings
S18 References                      → citation lookups, "who wrote X", source papers

Cross-cutting rules:
- "win rate / which system is better"      → S13 (architecture) + S16 (discussion)
- "cost / how much does this cost"         → S12 (preprocessing tables) + S15 (per-query cost)
- "MRR / Recall / retrieval metrics"       → S11 (definitions) + S14 (actual results)
- "why did hierarchical/PageIndex lose"    → S13 (failure cases) + S16 (TOC bottleneck)
- "reranking parameters / k values"        → S10 (method) + S14 (Table 4 results)
- "benchmark / dataset details"            → S7 (dataset construction)
"""

In [80]:
def llm_tree_search_with_expert(query: str, tree: list, expert_rules: str, model: str = "openai/gpt-oss-20b") -> dict:
    """
    Same as llm_tree_search() but with domain expert rules injected.
    The LLM uses these rules to guide its reasoning.
    """

    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out

    compressed_tree = compress(tree)

    prompt = f"""You are a domain expert analyzing a document.
Find all node IDs that most likely contain the answer to the query.
Use the expert routing rules below to guide your reasoning.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Expert Routing Rules (follow these carefully):
{PAPER_EXPERT_RULES}

Reply ONLY in this JSON format:
{{
  "thinking": "<your reasoning, referencing the expert rules>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    return call_llm(prompt=prompt, model=model)

In [83]:
# ── Test expert-guided retrieval ─────────────────────────────────────────────
query = "Why did the node-based system underperform?"

print(f"Query: {query}\n")

# Without expert rules
print("── Without Expert Rules ──")
basic   = llm_tree_search(query, tree)
print("Nodes:", json.loads(basic).get("node_list"))

print()

# With expert rules
print("── With Expert Rules ──")
guided  = llm_tree_search_with_expert(query, tree, PAPER_EXPERT_RULES)
print("Nodes:", json.loads(guided).get("node_list"))
print("Reasoning:", json.loads(guided).get("thinking", "")[:300])

Query: Why did the node-based system underperform?

── Without Expert Rules ──
Nodes: ['0011', '0015', '0010']

── With Expert Rules ──
Nodes: ['0013', '0015']
Reasoning: The query asks why the node-based system underperformed. According to the cross‑cutting rules, explanations for hierarchical or PageIndex failures are found in the Architecture Comparison section (S13) and the Discussion section that analyzes why vector systems beat hierarchical ones (S16). These co


In [84]:
# ── Full expert-guided RAG ───────────────────────────────────────────────────
def expert_rag(query: str, tree: list, node_map: dict, rules: str) -> str:
    """Expert-guided end-to-end RAG pipeline."""
    result  = llm_tree_search_with_expert(query, tree, rules)
    nodes   = find_nodes_by_ids(node_map=node_map, target_ids=json.loads(result).get("node_list", []))
    return generate_answer(query, nodes)

answer = expert_rag(
    query="Why did the node-based system underperform?",
    tree=tree,
    node_map=node_map,
    rules=PAPER_EXPERT_RULES
)
print(answer)

The node‑based system underperformed mainly because its structured traversal was limited by two factors:  

1. **Context‑window constraints** that caused 2 failed answers and 2 incorrect responses (5.1 | Page 6).  
2. A **retrieval bottleneck at the table‑of‑contents level**, where the LLM struggled to select relevant sections for complex financial queries, leading to poorer semantic matching compared to vector‑based retrieval (5.1 | Page 6).


In [85]:
# ── Delete document from cloud ───────────────────────────────────────────────

pi_client.delete_document(doc_id)
print(f"Deleted document: {doc_id}")

Deleted document: pi-cmrap6ixg023f01o3xrazyrxh
